In [ ]:
# ==============================================================
# 07 – Hierarchical Multi-Agent Training (Very Innovative)
# Title: Multi-Agent DRL + CNN Alternative Data for Credit Decisioning
# Core of RQ1 + supports RQ4
# Production-bank ready
# ==============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Load Fused Features
# --------------------------------------------------------------
X = np.load(DATA_PROCESSED / "X_train_fused.npy").astype(np.float32)
y = np.load(DATA_PROCESSED / "y_train.npy")
thin = np.load(DATA_PROCESSED / "thin_train.npy")

state_dim = X.shape[1]
print(f"State dim: {state_dim} | Samples: {len(X)} | Default rate: {y.mean():.2%}")

# --------------------------------------------------------------
# 2. Credit Environment
# --------------------------------------------------------------
class CreditEnv:
    def __init__(self, X, y, thin):
        self.X, self.y, self.thin = X, y, thin
        self.n = len(X)
        self.action_dim = 3   # 0=Reject, 1=Approve, 2=Counter
        self.reset()

    def reset(self):
        self.idx = np.random.permutation(self.n)
        self.ptr = 0
        return self.X[self.idx[0]]

    def step(self, action):
        i = self.idx[self.ptr]
        label, is_thin = self.y[i], self.thin[i]

        # Multi-objective reward
        if action == 1:  # Approve
            reward = 1.2 if label == 0 else -5.0
            if label == 0 and is_thin: reward += 0.4          # Inclusion bonus
        elif action == 2:  # Counter
            reward = 0.7 if label == 0 else -2.8
        else:  # Reject
            reward = 0.5 if label == 1 else -1.1
            if label == 0 and is_thin: reward -= 0.3          # Penalty for excluding good thin-file

        self.ptr += 1
        done = self.ptr >= self.n
        next_state = self.X[self.idx[self.ptr]] if not done else np.zeros(state_dim, dtype=np.float32)
        return next_state, reward, done, {"label": label, "thin": is_thin}

# --------------------------------------------------------------
# 3. Specialized Agent
# --------------------------------------------------------------
class SpecializedAgent(nn.Module):
    def __init__(self, state_dim, hidden=128, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, action_dim)
        )
    def forward(self, x):
        return self.net(x)

# --------------------------------------------------------------
# 4. Attention Coordinator (Innovative)
# --------------------------------------------------------------
class AttentionCoordinator(nn.Module):
    def __init__(self, num_agents, action_dim=3):
        super().__init__()
        self.query = nn.Linear(action_dim, action_dim)
        self.key   = nn.Linear(action_dim, action_dim)
        self.value = nn.Linear(action_dim, action_dim)
        self.scale = action_dim ** 0.5
        self.out   = nn.Sequential(
            nn.Linear(action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, agent_logits):
        # agent_logits: [num_agents, batch, action_dim]
        x = agent_logits.permute(1, 0, 2)          # [batch, agents, dim]
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        attn = torch.softmax(torch.bmm(Q, K.transpose(1, 2)) / self.scale, dim=-1)
        out = torch.bmm(attn, V).mean(dim=1)      # [batch, dim]
        return self.out(out), attn

# --------------------------------------------------------------
# 5. Hierarchical Multi-Agent Model
# --------------------------------------------------------------
class HierarchicalMARL(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super().__init__()
        self.agent_names = ["Risk", "Affordability", "Macro", "Fairness", "Pricing"]
        self.agents = nn.ModuleDict({
            name: SpecializedAgent(state_dim, action_dim=action_dim)
            for name in self.agent_names
        })
        self.coordinator = AttentionCoordinator(len(self.agent_names), action_dim)
        self.value_head = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state):
        agent_outputs = torch.stack(
            [self.agents[name](state) for name in self.agent_names], dim=0
        )  # [num_agents, batch, action_dim]
        final_logits, attn = self.coordinator(agent_outputs)
        value = self.value_head(state).squeeze(-1)
        return final_logits, value, attn, agent_outputs

# --------------------------------------------------------------
# 6. PPO Training for Multi-Agent
# --------------------------------------------------------------
model = HierarchicalMARL(state_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=2.5e-4)

env = CreditEnv(X, y, thin)

NUM_EPISODES = 70
GAMMA = 0.99
CLIP_EPS = 0.2
EPOCHS = 4

episode_rewards = []
attn_history = []

print("\n=== Hierarchical Multi-Agent PPO Training Started ===")

for episode in range(1, NUM_EPISODES + 1):
    state = env.reset()
    done = False
    ep_reward = 0

    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []

    while not done:
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value, attn, _ = model(state_t)
            dist = Categorical(logits=logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)

        next_state, reward, done, info = env.step(action.item())

        states.append(state)
        actions.append(action.item())
        rewards.append(reward)
        log_probs.append(log_prob.item())
        values.append(value.item())
        dones.append(done)

        state = next_state
        ep_reward += reward

    # GAE
    returns = []
    gae = 0
    values = values + [0]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + GAMMA * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + GAMMA * 0.95 * (1 - dones[t]) * gae
        returns.insert(0, gae + values[t])

    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    advantages = returns - torch.tensor(values[:-1], dtype=torch.float32, device=device)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    states_t = torch.tensor(np.array(states), dtype=torch.float32, device=device)
    actions_t = torch.tensor(actions, dtype=torch.long, device=device)
    old_log_probs = torch.tensor(log_probs, dtype=torch.float32, device=device)

    # PPO Update
    for _ in range(EPOCHS):
        logits, values_pred, attn, agent_outs = model(states_t)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions_t)
        entropy = dist.entropy().mean()

        ratio = torch.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = F.mse_loss(values_pred, returns)
        loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

    episode_rewards.append(ep_reward)
    attn_history.append(attn.mean(dim=0).detach().cpu().numpy())  # average attention

    if episode % 10 == 0 or episode == 1:
        print(f"Episode {episode:3d}/{NUM_EPISODES} | Avg Reward (last 10): {np.mean(episode_rewards[-10:]):8.2f}")

# --------------------------------------------------------------
# 7. Save Everything (Production Ready)
# --------------------------------------------------------------
torch.save({
    "model_state": model.state_dict(),
    "agent_names": model.agent_names,
    "state_dim": state_dim
}, RESULTS / "hierarchical_marl_final.pt")

np.save(RESULTS / "marl_episode_rewards.npy", np.array(episode_rewards))
np.save(RESULTS / "marl_attention_history.npy", np.array(attn_history))

print("\n✓ Hierarchical MARL model saved → results/hierarchical_marl_final.pt")

# --------------------------------------------------------------
# 8. Visualization
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.6)
axes[0].plot(pd.Series(episode_rewards).rolling(10).mean(), color="red", lw=2)
axes[0].set_title("Hierarchical MARL – Learning Curve")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].grid(True)

# Average attention over agents
avg_attn = np.mean(attn_history, axis=0).mean(axis=0)  # mean over episodes & batch
axes[1].bar(model.agent_names, avg_attn[:len(model.agent_names)])
axes[1].set_title("Average Attention Weight per Agent")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(RESULTS / "marl_training_overview.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ 07_Multi_Agent_Training completed successfully.")
print("This is the core innovative component of your research (RQ1).")